# AethyxLM - Production Colab Training (T4 GPU)

**Architecture:** 14M params, 8L, 256D, 8H, 128ctx, 32k vocab
**Dataset:** TinyStories (local corpus.txt)
**Storage:** GitHub = code, Google Drive = checkpoints/logs, Colab = compute

---

In [ ]:
# ============================================================
# CELL 1: MOUNT GOOGLE DRIVE & SETUP PROJECT
# ============================================================
from google.colab import drive, files
import os, sys, subprocess, shutil, json, time, glob, signal

print('Mounting Google Drive...')
drive.mount('/content/drive', force_remount=True)

# Persistent directories on Drive
DRIVE_ROOT = '/content/drive/MyDrive/AethyxLM'
DRIVE_CKPT = os.path.join(DRIVE_ROOT, 'checkpoints')
DRIVE_LOGS = os.path.join(DRIVE_ROOT, 'logs')
DRIVE_TOK = os.path.join(DRIVE_ROOT, 'tokenizer')
DRIVE_DATA = os.path.join(DRIVE_ROOT, 'dataset')
DRIVE_CONFIG = os.path.join(DRIVE_ROOT, 'configs')

for d in [DRIVE_CKPT, DRIVE_LOGS, DRIVE_TOK, DRIVE_DATA, DRIVE_CONFIG]:
    os.makedirs(d, exist_ok=True)

print(f'[OK] Drive mounted: {DRIVE_ROOT}')
print(f'[OK] Checkpoints: {DRIVE_CKPT}')
print(f'[OK] Logs: {DRIVE_LOGS}')
print(f'[OK] Configs: {DRIVE_CONFIG}')

# ============================================================
# CLONE/PULL FROM GITHUB (code lives in Git)
# ============================================================
REPO_URL = 'https://github.com/aethyx-ai/AethyxLM.git'
LOCAL_ROOT = '/content/AethyxLM'

if os.path.exists(os.path.join(LOCAL_ROOT, '.git')):
    print('Updating existing repo...')
    subprocess.run(['git', '-C', LOCAL_ROOT, 'pull'], check=True)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO_URL, LOCAL_ROOT], check=True)

# Fix nested directory from git clone (repo clones to /content/AethyxLM/AethyxLM)
nested = os.path.join(LOCAL_ROOT, 'AethyxLM')
if os.path.exists(nested):
    for item in os.listdir(nested):
        src = os.path.join(nested, item)
        dst = os.path.join(LOCAL_ROOT, item)
        if os.path.exists(dst):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            else:
                os.remove(dst)
        shutil.move(src, LOCAL_ROOT)
    os.rmdir(nested)

os.chdir(LOCAL_ROOT)

# Fix nested directory from git clone (repo clones to /content/AethyxLM/AethyxLM)
nested = os.path.join(LOCAL_ROOT, 'AethyxLM')
if os.path.exists(nested):
    for item in os.listdir(nested):
        src = os.path.join(nested, item)
        dst = os.path.join(LOCAL_ROOT, item)
        if os.path.exists(dst):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            else:
                os.remove(dst)
        shutil.move(src, LOCAL_ROOT)
    os.rmdir(nested)

sys.path.insert(0, LOCAL_ROOT)

print(f'[OK] Project: {LOCAL_ROOT}')
print(f'[OK] Config: {os.path.exists("configs/train_config.json")}')
print(f'[OK] Corpus: {os.path.exists("tokenizer/data/corpus.txt")}')

# Install deps
!pip install tokenizers datasets tensorboard -q

In [ ]:
# ============================================================
# CELL 2: VERIFY CUDA (accepts any CUDA GPU)
# ============================================================
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not available! Enable GPU: Runtime -> Change runtime type -> GPU')

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {device_name} ({vram_gb:.1f} GB)')

# Accept any CUDA GPU - just warn if unexpected
known_gpus = ['T4', 'L4', 'A100', 'V100', 'P100']
if not any(g in device_name for g in known_gpus):
    print(f"Warning: GPU '{device_name}' not in common Colab types {known_gpus}. Proceeding anyway...")

In [ ]:
# ============================================================
# CELL 4: TRAIN BPE TOKENIZER (32k vocab) + SAVE TO DRIVE
# ============================================================
import subprocess, sys, shutil, os, time, json

# Fix nested directory from git clone (repo clones to /content/AethyxLM/AethyxLM)
nested = os.path.join(LOCAL_ROOT, 'AethyxLM')
if os.path.exists(nested):
    for item in os.listdir(nested):
        src = os.path.join(nested, item)
        dst = os.path.join(LOCAL_ROOT, item)
        if os.path.exists(dst):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            else:
                os.remove(dst)
        shutil.move(src, LOCAL_ROOT)
    os.rmdir(nested)

os.chdir(LOCAL_ROOT)
sys.path.insert(0, LOCAL_ROOT)

print('Training tokenizer...')
result = subprocess.run(
    [sys.executable, '-m', 'tokenizer.train_tokenizer'],
    cwd=LOCAL_ROOT, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Tokenizer training failed')

# Verify
sys.path.insert(0, LOCAL_ROOT)
from tokenizer.tokenizer import AethyxTokenizer
tok = AethyxTokenizer()
print(f'[OK] Vocab size: {tok.vocab_size}')
ids = tok.encode('Hello world')
print(f'[OK] Encode: {ids}')
print(f'[OK] Decode: {tok.decode(ids)}')

# Robust Fix: If metadata.json was not created by the cloned train_tokenizer.py, generate it now!
metadata_path = 'tokenizer/metadata.json'
if not os.path.exists(metadata_path):
    print('Creating missing metadata.json dynamically...')
    corpus_path = 'tokenizer/data/corpus.txt'
    metadata = {
        'vocab_size': tok.vocab_size,
        'tokenizer_type': 'BPE',
        'special_tokens': ['<PAD>', '<UNK>', '<BOS>', '<EOS>'],
        'normalizer': {
            'type': 'Sequence',
            'components': [
                {'type': 'NFD'},
                {'type': 'Lowercase'},
                {'type': 'StripAccents'}
            ]
        },
        'pre_tokenizer': {
            'type': 'ByteLevel'
        },
        'trainer': {
            'type': 'BpeTrainer',
            'vocab_size': tok.vocab_size,
            'min_frequency': 2,
            'special_tokens': ['<PAD>', '<UNK>', '<BOS>', '<EOS>']
        },
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'dataset_used': corpus_path,
        'corpus_size_bytes': os.path.getsize(corpus_path) if os.path.exists(corpus_path) else 0,
    }
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

# Copy tokenizer and metadata safely to Drive
shutil.copy2('tokenizer/tokenizer.json', os.path.join(DRIVE_TOK, 'tokenizer.json'))
if os.path.exists(metadata_path):
    shutil.copy2(metadata_path, os.path.join(DRIVE_TOK, 'metadata.json'))
print('[OK] Tokenizer and metadata saved to Drive')


In [ ]:
# ============================================================
# CELL 4: TRAIN BPE TOKENIZER (32k vocab) + SAVE TO DRIVE
# ============================================================
import subprocess, sys, shutil, os, time, json

print('Training tokenizer...')
result = subprocess.run(
    [sys.executable, '-m', 'tokenizer.train_tokenizer'],
    cwd=LOCAL_ROOT, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Tokenizer training failed')

# Verify
sys.path.insert(0, LOCAL_ROOT)
from tokenizer.tokenizer import AethyxTokenizer
tok = AethyxTokenizer()
print(f'[OK] Vocab size: {tok.vocab_size}')
ids = tok.encode('Hello world')
print(f'[OK] Encode: {ids}')
print(f'[OK] Decode: {tok.decode(ids)}')

# Robust Fix: If metadata.json was not created by the cloned train_tokenizer.py, generate it now!
metadata_path = 'tokenizer/metadata.json'
if not os.path.exists(metadata_path):
    print('Creating missing metadata.json dynamically...')
    corpus_path = 'tokenizer/data/corpus.txt'
    metadata = {
        "vocab_size": tok.vocab_size,
        "tokenizer_type": "BPE",
        "special_tokens": ["<PAD>", "<UNK>", "<BOS>", "<EOS>"],
        "normalizer": {
            "type": "Sequence",
            "components": [
                {"type": "NFD"},
                {"type": "Lowercase"},
                {"type": "StripAccents"}
            ]
        },
        "pre_tokenizer": {
            "type": "ByteLevel"
        },
        "trainer": {
            "type": "BpeTrainer",
            "vocab_size": tok.vocab_size,
            "min_frequency": 2,
            "special_tokens": ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
        },
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "dataset_used": corpus_path,
        "corpus_size_bytes": os.path.getsize(corpus_path) if os.path.exists(corpus_path) else 0,
    }
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

# Copy tokenizer and metadata safely to Drive
shutil.copy2('tokenizer/tokenizer.json', os.path.join(DRIVE_TOK, 'tokenizer.json'))
if os.path.exists(metadata_path):
    shutil.copy2(metadata_path, os.path.join(DRIVE_TOK, 'metadata.json'))
print('[OK] Tokenizer and metadata saved to Drive')


In [ ]:
# ============================================================
# CELL 5: CONFIG - IMMUTABLE ORIGINAL + COLAB COPY
# ============================================================
import json

# Read original config (never modified)
with open('configs/train_config.json') as f:
    cfg = json.load(f)

# T4-optimized settings
cfg['training'].update({
    'max_steps': 20000,
    'warmup_steps': 2000,
    'batch_size': 32,
    'grad_accum_steps': 1,
    'use_amp': True,
    'eval_interval': 1000,
    'save_interval': 500,
    'log_interval': 100,
    'learning_rate': 3e-4,
    'grad_clip': 1.0,
    'weight_decay': 0.1,
    'min_lr_ratio': 0.1,
    'generate_interval': 1000,
})

# Write to COLAB config (never touches original)
colab_config_path = 'configs/train_config_colab.json'
with open(colab_config_path, 'w') as f:
    json.dump(cfg, f, indent=2)

# Backup both to Drive
shutil.copy2('configs/train_config.json', os.path.join(DRIVE_CONFIG, 'train_config.json'))
shutil.copy2(colab_config_path, os.path.join(DRIVE_CONFIG, 'train_config_colab.json'))

print('[OK] Config written to:')
print(f'  Original (unchanged): configs/train_config.json')
print(f'  Colab override: {colab_config_path}')
for k, v in cfg['training'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# CELL 6: AUTO-RESUME FROM DRIVE CHECKPOINT
# ============================================================
import glob

def find_latest_checkpoint():
    """Find latest checkpoint in Drive or local."""
    candidates = [
        os.path.join(DRIVE_CKPT, 'checkpoint_latest.pt'),
        'checkpoints/checkpoint_latest.pt',
    ]
    for base in [DRIVE_CKPT, 'checkpoints']:
        if os.path.exists(base):
            steps = sorted(glob.glob(os.path.join(base, 'checkpoint_step_*.pt')))
            if steps:
                candidates.append(steps[-1])

    for c in candidates:
        if os.path.exists(c):
            return c
    return None

resume_path = find_latest_checkpoint()
if resume_path:
    print(f'[OK] Found checkpoint: {resume_path}')
    RESUME_ARGS = ['--resume', resume_path]
else:
    print('[OK] No checkpoint found, starting fresh')
    RESUME_ARGS = []

In [ ]:
# ============================================================
# CELL 7: SYNC CHECKPOINTS + LOGS + CONFIG (LOCAL <-> DRIVE)
# ============================================================
def sync_to_drive():
    """Copy local checkpoints, logs, config to Drive."""
    # Checkpoints
    if os.path.exists('checkpoints'):
        for f in os.listdir('checkpoints'):
            if f.endswith('.pt'):
                try:
                    shutil.copy2(os.path.join('checkpoints', f),
                               os.path.join(DRIVE_CKPT, f))
                except Exception as e:
                    print(f'  Sync failed for {f}: {e}')

    # Logs
    if os.path.exists('logs'):
        for f in os.listdir('logs'):
            try:
                shutil.copy2(os.path.join('logs', f),
                               os.path.join(DRIVE_LOGS, f))
            except Exception as e:
                print(f'  Log sync failed for {f}: {e}')

    # Config (colab version)
    colab_cfg = 'configs/train_config_colab.json'
    if os.path.exists(colab_cfg):
        try:
            shutil.copy2(colab_cfg, os.path.join(DRIVE_CONFIG, 'train_config_colab.json'))
        except Exception as e:
            print(f'  Config sync failed: {e}')

def sync_from_drive():
    """Copy Drive checkpoints to local before training."""
    if not os.path.exists(DRIVE_CKPT):
        return
    os.makedirs('checkpoints', exist_ok=True)
    for f in os.listdir(DRIVE_CKPT):
        if f.endswith('.pt'):
            src = os.path.join(DRIVE_CKPT, f)
            dst = os.path.join('checkpoints', f)
            if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
                try:
                    shutil.copy2(src, dst)
                    print(f'  Synced from Drive: {f}')
                except Exception as e:
                    print(f'  Sync failed for {f}: {e}')

# Initial sync from Drive
sync_from_drive()
print('[OK] Sync ready (checkpoints + logs + config)')

In [ ]:
# ============================================================
# CELL 8: TRAINING WRAPPER WITH TRY/FINALLY + AUTO-SYNC
# ============================================================
import torch
import threading, time

print('Starting training on', torch.cuda.get_device_name(0))
print('=' * 60)

cmd = [sys.executable, 'train.py',
       '--config', 'configs/train_config_colab.json',
       '--device', 'cuda']

if RESUME_ARGS:
    cmd.extend(RESUME_ARGS)

print(f'Command: {" ".join(cmd)}')
print('-' * 60)

stop_sync = False
sync_lock = threading.Lock()

def periodic_sync():
    while not stop_sync:
        time.sleep(300)
        if not stop_sync:
            with sync_lock:
                sync_to_drive()
                print(f'[{time.strftime("%H:%M:%S")}] Synced checkpoints + logs + config to Drive')

sync_thread = threading.Thread(target=periodic_sync, daemon=True)
sync_thread.start()

start = time.time()
try:
    result = subprocess.run(cmd, cwd=LOCAL_ROOT)
finally:
    stop_sync = True
    sync_thread.join(timeout=10)
    sync_to_drive()
    elapsed = time.time() - start
    print('=' * 60)
    print(f'Training finished in {elapsed/3600:.1f}h')
    print(f'Exit code: {result.returncode}')

if result.returncode == 0:
    print('[OK] Training completed successfully!')
else:
    print(f'[FAIL] Training failed with code {result.returncode}')
    print('[INFO] You can resume from last checkpoint on next session')

In [ ]:
# ============================================================
# CELL 9: DOWNLOAD FINAL CHECKPOINTS + LOGS
# ============================================================
from google.colab import files

ckpt_best = 'checkpoints/checkpoint_best.pt'
ckpt_latest = 'checkpoints/checkpoint_latest.pt'
ckpt_steps = sorted([f for f in os.listdir('checkpoints') 
                   if f.startswith('checkpoint_step_')])

for f in [ckpt_best, ckpt_latest] + ckpt_steps:
    if os.path.exists(f):
        print(f'Downloading: {f}')
        files.download(f)
    else:
        print(f'Not found: {f}')

# Also download logs if they exist
if os.path.exists('logs'):
    for f in os.listdir('logs'):
        files.download(os.path.join('logs', f))

In [ ]:
# ============================================================
# CELL 10: QUICK INFERENCE TEST
# ============================================================
import torch
from model.gpt import GPT
from tokenizer.tokenizer import AethyxTokenizer

device = 'cuda'
model = GPT().to(device)
tok = AethyxTokenizer()

ckpt_path = 'checkpoints/checkpoint_best.pt'
if not os.path.exists(ckpt_path):
    ckpt_path = 'checkpoints/checkpoint_latest.pt'

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

@torch.no_grad()
def generate(prompt, max_new=200, temp=0.8, top_k=50):
    ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new):
        logits = model(ids[:, -128:])
        logits = logits[:, -1, :] / temp
        if top_k > 0:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('inf')
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)
        ids = torch.cat([ids, next_id], dim=1)
    return tok.decode(ids[0].tolist())

print('Sample generation:')
print('-' * 60)
print(generate('Once upon a time'))
print('-' * 60)
print(generate('The little boy'))
print('-' * 60)
print(generate('In a magical forest'))